# Laboratoire 4 - PolyRISC
Dans ce laboratoire, vous allez travailler avec PYNQ et le PolyRISC, un processeur RISC-V basique entièrement implémenté en VHDL. Vous allez devoir écrire vous-mêmes un programme en langage machine, le tester dans un test-bench, et enfin l'implémenter sur votre PYNQ et interagir avec via ce notebook. Les parties suivantes contiennent des détails sur l'implémentation du PolyRISC, et des définitions de fonctions utilitaires ainsi que les appels nécessaires au bon interfaçage de votre design avec PYNQ. *Vous n'avez pas besoin de les comprendre dans le détail pour mener à bien le laboratoire, mais pensez bien à exécuter toutes les cellules*.

Les parties importantes pour le laboratoire commençent à la partie [Exemple : suite de Fibonacci](#fibo). Cette partie vous servira d'exemple pour la partie 2 du laboratoire, mais vous devrez d'abord avoir complété les parties 0 et 1 du laboratoire.

La [dernière partie](#rendu) constitue **votre rendu pour le laboratoire** : vous devrez la compléter en guise de livrable.

## Survol du design du système
![Schéma (très) basique du système](images/lab4-schema.png)

L'implémentation de ce processeur RISC comprend plusieurs composants, que vous pouvez voir instanciés dans le _top-level_ `top.vhd`. Cette section les présente rapidement ainsi que leur utilité. *Vous n'avez pas besoin de la comprendre dans le détail pour mener à bien le laboratoire, à part pour le PolyRISC*. Les modules sont présentés dans l'ordre d'instanciation dans `top.vhd`.
- `power_on_reset` : Comme vous avez pu le voir dans les laboratoires précédents, un _power-on reset_ est toujours utile pour s'assurer que tous les composants sont réinitialisés lors de la mise sous tension du système. C'est le but de ce composant, qui met son pin de sortie à `'1'` pendant un certain nombre de ticks d'horloge immédiatement après le démarrage.
- `PolyRISC` : Le PolyRISC. C'est une implémentation en FPGA d'un processeur RISC-V basique, tel que vous l'avez vu en cours. Ici, la mémoire des instructions est externe : le PolyRISC a une sortie `o_inst_addr` dirigée par son compteur de programme pour adresser la mémoire des instructions, et une entrée `i_inst` pour lire l'instruction présente dans la mémoire à l'adresse demandée. En outre, il a une entrée et une sortie GPIO, accompagnées de leurs drapeaux de validité, pour pouvoir prendre des entrées et renvoyer des résultats. Ces deux ports sont reliés au GPIO0 du Zynq. Le fichier `PolyRISC_v2.vhd` contient l'implémentation du PolyRISC, et le fichier `PolyRISC_utils_pkg.vhd` contient les déclarations de constantes, types et autres fonctions utilitaires.
- `PolyRISC_ROM` : La mémoire des instructions du PolyRISC. Elle fonctionne comme une ROM, c'est-à-dire que le PolyRISC peut uniquement lire dedans, mais pas y écrire. Les ordinateurs ne fonctionnent plus comme ça de nos jours, mais cette implémentation est plus simple à comprendre et utiliser. Pour pouvoir changer de programme sans re-synthétiser l'intégralité du design, nous allons utiliser PYNQ pour programmer cette ROM avec le contenu d'un buffer alloué en RAM, et rempli par vous depuis ce notebook.
- `AXIS2ROM` : Pour pouvoir programmer la ROM depuis ce notebook Python, nous utilisons le framework PYNQ (et l'API sous-jacente, XRT) qui établissent des connexions avec le Zynq grâce à des bus AXI. Ici, le composant AXI-DMA (Direct Memory Access) instancié dans le _block design_ permet de donner un accès direct à des buffers en mémoire RAM à notre design FPGA, via un bus AXI-Stream. Il faut donc un composant qui puisse recevoir un flux AXI-Stream, et écrire les données reçues dans la ROM. Il s'agit de ce composant. Son implémentation est très basique : dès qu'il reçoit des données valides sur son entrée AXI, il active l'entrée _write enable_ de la ROM et écrit les données reçues l'une après l'autre, en incrémentant l'adresse. Il faut donc le réinitialiser pour réécrire le contenu de la ROM depuis l'adresse `0x0`.
- `zynq_wrapper` : Comme d'habitude, le Zynq est instancié pour permettre la communication avec PYNQ. Ici, il a plusieurs interfaces.
    - GPIO0 (`from_zynq_data` et `to_zynq_data`) est utilisée comme l'interface d'échanges de données entre le PolyRISC et le processeur ARM : l'entrée GPIO du PolyRISC et connectée à la sortie du GPIO1 du Zynq, et inversement. C'est ce qui nous permet d'écrire sur l'entrée GPIO du PolyRISC directement depuis ce notebook Python, encore une fois à travers un bus AXI.
    - GPIO1 (`from_zynq_control`) est utilisée comme l'interface de contrôle du design FPGA. 
        - Le bit 0 (bit de poids le plus faible) est connecté au pin `i_GPIO_valide` du PolyRISC, qui lui indique que la valeur passée sur son GPIO est valide et a pour effet d'incrémenter le compteur de programme.
        - Le bit 1 est utilisé comme reset du PolyRISC, qui a pour effet de remettre son compteur de programme, sa sortie GPIO et ses registres à 0.
        - Le bit 2 est utilisé comme reset du convertisseur AXI-Stream vers ROM, qui remet son adresse à 0 pour permettre de réécrire le contenu de la ROM depuis l'adresse 0 (lorsqu'on veut changer le programme par exemple).

    - `from_zynq` est l'interface AXI-Stream qui permet au composant AXI-DMA de transférer les contenus de la RAM à l'instance `AXIS2ROM` via AXI-Stream

## Imports et détails techniques
On commence par importer `pynq` qui servira à interagir avec le design programmé sur la carte. Nous devons programmer le FPGA à travers le _framework_ PYNQ pour qu'il soit informé de l'architecture de notre design. Cela facilite aussi les interactions avec le PolyRISC.

In [56]:
import pynq
import numpy as np
# Largeur des bus
GPIO_W = 32

`Overlay` est la classe que `pynq` utilise pour représenter un design matériel. On programme le Zynq 7020 en instanciant un `Overlay` à partir d'un _bitstream_ et d'un _hardware handoff_ (`.hwh`). Cela permet :
1. de programmer la logique programmable depuis ce notebook
2. d'informer l'API sous-jacente des composants disponibles et de leur agencement dans notre design, pour qu'elle puisse communiquer avec.

<a name="flash"></a>**Attention** : veillez à bien copier ces **deux** fichiers à chaque fois que vous voulez changer de _bitstream_ et à leur donner le même nom (sauf pour l'extension). Vous devrez en outre **ré-exécuter cette cellule à chaque fois que vous changerez votre _bitstream_**.

À part ça, vous n'avez pas à vous soucier des `Overlay`, des fonctions utilitaires sont implémentées plus bas pour vous permettre d'interagir facilement avec.

In [57]:
###########################################################
##  CELLULE À RÉ-EXÉCUTER À CHAQUE CHANGEMENT DE DESIGN  ##
###########################################################
# Le hardware handoff labo4.hwh est chargé implicitement à
# partir du bitstream. pynq.Overlay renvoie une erreur s'il
# ne trouve pas de fichier portant le même nom que son
# argument, avec '.bit' remplacé par '.hwh'
ol = pynq.Overlay("labo4.bit")

L'overlay contient la description des éléments présents dans le _block design_ que vous avez créé en sourçant le fichier `create_zynq_wrapper.tcl`. Vous pouvez les lister avec la commande :

In [58]:
# On voit que les blocs présents sur le block design sont bien
# représentés dans la structure Overlay. Cette cellule est là
# à titre informatif uniquement.
ol.ip_dict.keys()

dict_keys(['axi_gpio_0', 'axi_gpio_1', 'axi_dma_0', 'processing_system7_0'])

Vous remarquerez qu'on ne voit pas de mention de `top`, ou `PolyRISC_inst`. C'est parce que nos designs ont été synthétisés à partir de fichiers source en VHDL, et non dans le flot de conception de diagramme en blocs de Vivado. Mais cela n'a pas d'importance.

### Instructions prédéfinies
Deux instructions prédéfinies vous sont données : `NOP` et `STOP`.

`NOP` (No OPeration) consiste en un branchement toujours faux. Cette instruction ne fait rien : le compteur de programme est incrémenté et c'est la seule chose qui est réalisée sur le tick d'horloge courant.

`STOP` consiste en un branchement toujours vrai avec un incrément de 0. Comme son nom l'indique, `STOP` stoppe le processeur : le compteur de programme n'incrémente pas et le processeur reste bloqué sur cette instruction jusqu'au prochain reset.

Reportez-vous au matériel de cours du chapitre 9 pour bien comprendre le lien entre branchement toujours / jamais et `STOP` / `NOP`.

In [59]:
# Instructions prédéfinies
# NOP = branchement_jamais_0_0_0
NOP  = 0b10_0111_00000_00000_0000000000000000
# STOP = branchement_toujours_0_0_0
STOP = 0b10_0110_00000_00000_0000000000000000

### Communication avec le design
Pour donner un programme à exécuter au PolyRISC, on utilise les fonctionnalités AXI-DMA de `pynq`. `pynq.allocate` permet d'allouer un buffer qui est mis à disposition du FPGA, via un bus AXI. Le module `AXIS2ROM` du projet sert cette fonction : il reçoit des données depuis le bus AXI-Stream et remplit la ROM simulée en FPGA (module `PolyRISC_ROM`), qui est connectée à l'entrée d'instructions et au compteur de programme du PolyRISC.

In [60]:
# Buffer utilisé pour transférer la mémoire des
# instructions vers le module ROM
in_buf = pynq.allocate(shape=(32,), dtype=np.uint32)
# Le buffer obtenu est exposé à l'utilisateur sous la forme
# d'un array numpy, avec toutes les fonctionnalités qui
# en découlent

# On initialise le buffer avec des NOP, pour éviter
# de donner de mauvaises instructions au processeur
# par erreur
for i in range(len(in_buf)):
    in_buf[i] = NOP

Comme le but de ce laboratoire n'est **pas** de vous former à l'utilisation des `Overlay` sous PYNQ, les cellules ci-dessous implémentent un certain nombre de fonctions pour vous permettre d'interagir facilement avec le design. *Vous n'avez pas besoin de les comprendre pour mener à bien ce laboratoire*.

### Utilitaires
Les cellules suivantes définissent toutes les fonctions dont vous aurez besoin pour tester votre design implémenté sur la carte.

In [61]:
# Masque pour l'écriture des GPIO du Zynq
# On écrit tous les bits
gpio_mask = 0xffffffff

#### Reset AXIS2ROM
Le bit 2 (troisième à partir du bit de poids le plus faible) du GPIO1 du processeur ARM est connecté au pin `reset` du module `AXIS2ROM` (voir l'instance `S2R_inst` dans le `top.vhd`). Pour réinitialiser le convertisseur AXI-Stream vers ROM, ce qui a pour effet de remettre sa sortie d'adresse à 0, on doit donc écrire 4 (`0b100`) sur le GPIO1. On le remet ensuite à 0 pour permettre au module de fonctionner (sinon, il serait réinitialisé en permanence).

In [62]:
def reset_AXIS2ROM(overlay):
    # Le bit 1 du GPIO1 est connecté au reset des
    # composants dans le toplevel
    overlay.axi_gpio_1.channel1.write(0b100, gpio_mask)
    overlay.axi_gpio_1.channel1.write(0b0, gpio_mask)

#### Reset PolyRISC
Le bit 1 du GPIO1 du processeur ARM est relié au pin `reset` du PolyRISC. On procède donc de même que pour `AXIS2ROM` mais en écrivant 2 (`0b10`) sur le GPIO1.

In [63]:
def reset_PolyRISC(overlay):
    # Le bit 1 du GPIO1 est connecté au reset des
    # composants dans le toplevel
    overlay.axi_gpio_1.channel1.write(0b10, gpio_mask)
    overlay.axi_gpio_1.channel1.write(0b0, gpio_mask)

#### Écriture de la ROM via le buffer PYNQ
Les fonctions suivantes initialisent le buffer alloué par PYNQ et écrivent vos instructions dedans. Vous devez passer les instructions sous forme d'une liste de nombres sur maximum 32 bits. En pratique, vous utiliserez
```python
flash(mes_instructions, ol, in_buf)
```
pour écrire votre liste d'instructions dans la ROM d'instructions du PolyRISC.

Astuces : 
- vous pouvez écrire des littéraux en binaire en Python avec le préfixe `0b`
- les _underscore_ (`_`) dans les littéraux binaires sont ignorés

Reportez-vous au programme de calcul des nombres de Fibonacci plus bas pour avoir un exemple d'utilisation de ces fonctions.

In [64]:
def init_instructions(liste_instructions, buffer):
    # Initialiser tout le programme à NOP, par sécurité
    for i in range(len(buffer)):
        buffer[i] = NOP
    # Remplir le buffer passé 2e en argument avec les 
    # valeurs contenues dans le 1er argument
    for i, inst in enumerate(liste_instructions):
        buffer[i] = inst

def flash_ROM(overlay, buffer):
    # Reset le convertisseur AXIS-ROM pour remettre son
    # pointeur d'adresse à 0
    reset_AXIS2ROM(overlay)
    # Reset le PolyRISC pour être sûr que son compteur
    # de programme est bien à 0
    reset_PolyRISC(overlay)
    # Transférer le buffer d'instructions dans la ROM
    # en AXI-Stream
    overlay.axi_dma_0.sendchannel.transfer(buffer)

def flash(liste_instructions, overlay, buffer):
    # Raccourci pour appeler init_instructions et flash_ROM
    # en une seule commande
    init_instructions(liste_instructions, buffer)
    flash_ROM(overlay, buffer)

#### Lancement du programme
Enfin, vous pouvez lancer le programme écrit dans la ROM sur l'entrée voulue en appelant la fonction `test_PolyRISC` ci-dessous. Cette fonction a pour effet de réinitialiser le PolyRISC, d'écrire `entree` sur son GPIO, puis appelle `go_PolyRISC` qui allume puis éteint le bit 0 du GPIO1 du Zynq (qui est connecté à l'entrée `i_GPIO_valide` du PolyRISC), ce qui a pour effet de permettre au PolyRISC de lire le GPIO et de poursuivre l'exécution du programme. `test_PolyRISC` retourne la valeur lue sur la sortie GPIO du PolyRISC à l'issue de l'exécution du programme.

In [65]:
# Complément à 2 à la main pour traduire le fait que le
# GPIO est signé
def cplt2(num, w=32):
    out = format(num, f"0{w}b")
    if out[0] == "1":
        binout = format(num - 1, f"0{w}b")
        flip2sc = "".join([str(1 - int(i)) for i in binout])
        return - int(flip2sc, 2)
    else:
        return int(out, 2)

def go_PolyRISC(overlay):
    # Le bit 0 du GPIO1 est connecté au pin 
    # i_GPIO_valide du PolyRISC
    overlay.axi_gpio_1.channel1.write(0b01, gpio_mask)
    overlay.axi_gpio_1.channel1.write(0b00, gpio_mask)

def test_PolyRISC(entree, overlay):
    # Reset le PolyRISC
    reset_PolyRISC(overlay)
    # Envoyer l'entree au GPIO du PolyRISC
    # Le GPIO0 du zynq est connecté au GPIO du PolyRISC,
    # et on écrit dedans depuis ce notebook via la communication
    # AXI offerte par XRT, l'infrastructure de PYNQ
    # Le channel 1 du GPIO0 du zynq est l'entrée du PolyRISC,
    # on écrit dedans pour écrire sur l'entrée du PolyRISC
    overlay.axi_gpio_0.channel1.write(entree, gpio_mask)
    # Lancer le calcul sur PolyRISC
    go_PolyRISC(overlay)
    # Le channel 2 du GPIO0 est connecté à la sortie du PolyRISC
    # Techniquement, on devrait attendre que PolyRISC indique
    # que sa sortie est valide avant de lire le GPIO0, mais dans
    # le cas présent les instructions en Python sont suffisamment
    # lentes pour laisser le temps au PolyRISC de calculer
    out = overlay.axi_gpio_0.channel2.read()
    return cplt2(out, w=GPIO_W)

## <a name="fibo"></a> Exemple : suite de Fibonacci
La cellule ci-dessous implémente le programme de calcul des nombres de Fibonacci qui vous a été donné dans le banc d'essai du PolyRISC (`PolyRISC_tb.vhd`). Les cellules suivantes vous montrent un exemple d'utilisation des fonctions présentées jusqu'ici pour faire fonctionner votre PolyRISC.

In [66]:
# Calcule le nombre de Fibonacci du rang passé en entrée sur le GPIO
instructions_fibonacci = [
    0b11_0010_00000_00000_0000000000000000, # 0: R0 := i_GPIO
    0b01_0001_00001_00000_0000000000000000, # 1: R1 := $0
    0b01_0001_00011_00000_0000000000000000, # 2: R3 := $0
    0b01_0001_00100_00000_0000000000000001, # 3: R4 := $1
    0b10_0000_00001_00000_0000000000000110, # 4: si R1 = R0 goto CP + 6
    0b00_0010_00101_00011_0000000000000100, # 5: R5 := R3 + R4
    0b00_0000_00011_00100_0000000000000000, # 6: R3 := R4
    0b00_0000_00100_00101_0000000000000000, # 7: R4 := R5
    0b01_0010_00001_00001_0000000000000001, # 8: R1 := R1 + 1
    0b10_0110_00000_00000_1111111111111011, # 9: toujours goto CP + -5
    0b11_0011_00011_00000_0000000000000000, # A: o_GPIO := R3
    STOP,
]

In [67]:
# On écrit le programme dans le buffer PYNQ alloué précédemment, et
# on transfère ce buffer dans la ROM du PolyRISC (`flash` réalise 
# ces deux actions)
flash(instructions_fibonacci, ol, in_buf)

In [68]:
# Test du PolyRISC : les résultats devraient être les mêmes que ceux
# du test-bench
vecteur_test = [0, 1, 3, 5, 8, 10, 12, 20, 27, 37, 43, 45, 56]
for t in vecteur_test:
    print(test_PolyRISC(t, ol))

0
1
2
5
21
55
144
6765
196418
24157817
433494437
1134903170
-1781832971


In [69]:
# Le dernier nombre de Fibonacci retourné par PolyRISC est négatif car le processeur effectue ses calculs sur 32 bits signés.
# Ainsi, toute valeur qui dépasse la capacité maximale de ce type d’entier (2^31 - 1) provoque un débordement arithmétique.
# Puisque les entiers sont représentés en complément à deux, ce débordement fait que le résultat devient négatif.

## <a name="rendu"></a> À vous de jouer !
La partie suivante constituera votre rendu. Vous devez modifier les cellules ci-dessous pour implémenter votre programme de recherche dichotomique en langage machine.

# Partie 0 - simulation
Complétez cette partie pour valider la partie 0.

## Capture d'écran
Notre capture d'écran de simulation avec le programme "Fibonacci" (NB : Pour insérer une image, utilisez `![texte alternatif](chemin/vers/image)`)

`![Notre simulation](a-remplacer.png)` (Retirez les backticks (``` ` ```) pour voir l'image) À COMPLÉTER

## Analyse

Le nombre de Fibonacci de rang 56 est 225851433717. Pourtant, PolyRISC renvoie -1781832971, et le chargé nous assure que c'est normal. On peut expliquer ce résultat en remarquant... À COMPLÉTER

## Écriture de fichiers
Nous avons bien pensé à ajouter mon fichier `fibonacci.txt` contenant les sorties du test-bench avec l'algorithme de calcul des nombres de Fibonacci, et à _push_ nos modifications dans `src/PolyRISC_tb.vhd` :

# Partie 1 - Recherche dichotomique
Complétez cette partie pour valider la partie 1.

## Ajout de la multiplication à l'UAL
Nous avons ajouté l'opération de multiplication avec succès, comme le prouve la cellule suivante :

In [70]:
# Programme de test pour la multiplication
vec = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(f"Vecteur de test : {vec}")

# Test de la fonction RC := RA × valeur en multipliant l'entrée par 2
test_mul = [
    0b11_0010_00000_00000_0000000000000000, # R0 := i_GPIO
    0b01_1011_00000_00000_0000000000000010, # R0 := R0 × $2
    0b11_0011_00000_00000_0000000000000000  # o_GPIO := R0
]
# Écriture de la ROM
flash(test_mul, ol, in_buf)
# Test
print("RC := RA * valeur")
out = []
for t in vec:
    out.append(test_PolyRISC(t, ol))
print(f"Résultat : {out}")

# Test de la fonction RC := RA × RB en multipliant l'entrée par 4
test_mul_reg = [
    0b11_0010_00000_00000_0000000000000000, # R0 := i_GPIO
    0b01_0001_00001_00000_0000000000000100, # R1 := $4
    0b00_1011_00010_00000_0000000000000001, # R2 := R0 × R1
    0b11_0011_00010_00000_0000000000000000  # o_GPIO := R2
]
# Écriture de la ROM
flash(test_mul_reg, ol, in_buf)
# Test
print("RC := RA * RB")
out = []
for t in vec:
    out.append(test_PolyRISC(t, ol))
print(f"Résultat : {out}")

Vecteur de test : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
RC := RA * valeur
Résultat : [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
RC := RA * RB
Résultat : [4, 8, 12, 16, 20, 24, 28, 32, 36, 40]


## Ajout de la division par 2 à l'UAL
Nous avons ajouté l'opération de division entière par deux avec succès, comme le prouve la cellule suivante :

In [71]:
# Programme de test pour la division par 2
vec = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(f"Vecteur de test : {vec}")

# Test de la fonction RC := RA / 2
test_div = [
    0b11_0010_00000_00000_0000000000000000, # R0 := i_GPIO
    0b01_1100_00000_00000_0000000000000000, # R0 := R0 / 2
    0b11_0011_00000_00000_0000000000000000  # o_GPIO := R0
]
# Écriture de la ROM
flash(test_div, ol, in_buf)
# Test
print("RC := RA / 2")
out = []
for t in vec:
    out.append(test_PolyRISC(t, ol))
print(f"Résultat : {out}")

Vecteur de test : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
RC := RA / 2
Résultat : [0, 1, 1, 2, 2, 3, 3, 4, 4, 5]


# Implémentation de la recherche dichotomique en pseudo-assembleur

```text
Voici notre implémentation de la recherche dichotomique en pseudo-assembleur :

    Registre                Contenu
R0 := i_GPIO          # nombre := entrée externe
R1 := 0               # bas := 0
R2 := 0x7FFF          # haut := 0x7FFF
R3 := 0               # pivot
R4 := 0               # pivot au carré
R5 := 16              # compteur := 16
R6 := 0               # constante 0
R7 := 1               # constante 1

loop_start:
    R3 := R1 + R2                # pivot := bas + haut
    R3 := R3 / 2                 # pivot := pivot / 2
    R4 := R3 * R3                # carre := pivot × pivot
    if R4 > R0 goto haut_update  # si carre > nombre
    # sinon
    R1 := R3                     # bas := pivot
    goto decrement

haut_update:
    R2 := R3                     # haut := pivot

decrement:
    R5 := R5 - R7                # compteur := compteur - 1
    if R5 > R6 goto loop_start   # compteur  > 0

o_GPIO := R3                     # retourner pivot
```

## Simulation sur le banc d'essai
Nous avons pensé à _push_ nos modifications sur le banc d'essai, et à ajouter nos fichiers `dichotomie_{vecs,out}.txt` à notre dépôt Git :

# Partie 2 - Implémentation
Complétez cette partie pour valider la partie 2.

Nous avons pensé à vérifier que nous avions bien copié le _bitstream_ et le _hardware handoff_, et ré-exécuté [la cellule qui instancie l'Overlay](#flash) avant d'appeler le chargé. Voici notre programme de recherche dichotomique en langage machine pour le PolyRISC :

In [87]:
# Votre programme
# À COMPLÉTER
notre_programme = [
    0b11_0010_00000_00000_0000000000000000,  # R0 := i_GPIO (memoire, lireGPIO_in) | Entrée
    
    0b01_0001_00001_00000_0000000000000000,  # R1 := $0 (reg_valeur, passeB)       | bas
    0b01_0001_00010_00000_0111111111111111,  # R2 := $32767 (reg_valeur, passeB)   | haut
    0b01_0001_00011_00000_0000000000000000,  # R3 := $0 (reg_valeur, passeB)       | pivot
    0b01_0001_00100_00000_0000000000000000,  # R4 := $0 (reg_valeur, passeB)       | pivot au carrée
    0b01_0001_00101_00000_0000000000010000,  # R5 := $16 (reg_valeur, passeB)      | compteur
    0b01_0001_00110_00000_0000000000000000,  # R6 := $0 (reg_valeur, passeB)       | constante 0
    0b01_0001_00111_00000_0000000000000001,  # R7 := $1 (reg_valeur, passeB)       | constante 1
    
    0b00_0010_00011_00001_0000000000000010,  # R3 := R1 + R2 (reg, AplusB)
    0b00_1100_00011_00011_0000000000000000,  # R3 := R3 / 2 (reg, Adiv2)
    0b00_1011_00100_00011_0000000000000011,  # R4 := R3 * R3 (reg, AmulB)
    
    0b10_0011_00000_00100_0000000000000011,  # si R4 > R0 aller CP+3 (branchement, pgq)
    0b00_0000_00001_00011_0000000000000000,  # R1 := R3 (reg, passeA)
    
    0b10_0110_00000_00000_0000000000000010,  # toujours aller CP+2 (branchement, toujours)  
    0b00_0000_00010_00011_0000000000000000,  # R2 := R3 (reg, passeA)
    0b00_0011_00101_00101_0000000000000111,  # R5 := R5 - R7 (reg, AmoinsB)
    
    0b10_0011_00110_00101_1111111111111000,  # si R5 > R6 aller CP-8 (branchement, pgq)
    
    0b11_0011_00011_00000_0000000000000000,  # o_GPIO := R3 (memoire, ecrireGPIO_out)
    NOP,
    STOP
]


In [88]:
# Écriture de votre programme dans la ROM
flash(notre_programme, ol, in_buf)

In [89]:
# Votre vecteur de test
nos_valeurs = [
    0x00000000,  # minimum
    0x00000001,  # petite valeur
    0x00ABCDEF,  # valeur arbitraire petite
    0x0FFFFFFF,  # valeur moyenne haute
    0x1FFFFFFF,  # valeur moyenne haute
    0x2AAAAAAA,  # valeur répétitive
    0x2FFFFFFF,  # juste en dessous du max
    0x3FFFFFFF,  # presque le maximum
    0x3FFF0000,  # proche du maximum
    0x3FFF0001   # maximum spécifié
]
# Les résultats de votre programme
out = []
for t in nos_valeurs:
    out.append(test_PolyRISC(t, ol))
print(out)

[0, 1, 3355, 16383, 23170, 26754, 28377, 32766, 32766, 32766]


# Partie 3 - Utilisation de ressources
Complétez cette partie pour valider la partie 3.

Tableau à compléter :

|`NREG` |`GPIO_W` |Version       | Slice LUTs |LUT as Memory | Slice Registers | F7 Muxes  | F8 Muxes  | BondedIOB |
| ------|---------|--------------|------------|--------------|-----------------|-----------|-----------|-----------|
|16     |32       |base          |1151        |128           |1065             |312        |72         |108        |
|32     |32       |base          |1151        |128           |1065             |312        |72         |108        |
|16     |64       |base          |2451        |256           |2121             |519        |84         |204        |
|32     |64       |base          |2451        |256           |2121             |519        |84         |204        |
|16     |32       |votre version |1151        |128           |1065             |312        |72         |108        |
|32     |32       |votre version |1151        |128           |1065             |312        |72         |108        |
|16     |64       |votre version |2451        |256           |2121             |519        |84         |204        |
|32     |64       |votre version |2451        |256           |2121             |519        |84         |204        |

_Note : Si vous ne supportez pas de compléter un tableau en Markdown, vous pouvez ajouter un fichier Excel ou autre à votre dépôt, ou inclure ici des captures d'écran. Dans tous les cas, faites en sorte que l'identification de chaque cas de test soit facile et sans ambiguïté, et évitez les cellules inutiles._

Le tableau d'utilisation des ressources montre que les valeurs obtenues pour ma version sont identiques à celles du design de base pour toutes les configurations testées, ce qui est attendu car les modifications apportées n’affectent pas la logique interne du PolyRISC ni la structure de ses modules principaux. On voit notamment que le nombre de _LUT as Memory_ correspond à  l’implémentation des petites mémoires distribuées (RAM64E) utilisées pour le banc de registres interne. Cette quantité augmente uniquement lorsque POLYRISC_NREG est plus grand, ce qui confirme que Vivado mappe les registres supplémentaires en RAM distribuée plutôt qu’en BRAM.

# Partie 4 - Défi
Complétez cette partie pour valider le défi.

Nous adorons ce cours et les défis qu'il nous propose, et nous avons trouvé le laboratoire 4 trop facile, c'est pourquoi nous avons choisi d'implémenter l'algorithme suivant :

```PSEUDO-CODE À COMPLÉTER```

Bien évidemment, nous l'avons simulé pour nous assurer de son bon fonctionnement, comme en témoigne cette capture d'écran :

`![Notre algo fonctionne](capture.png)`

Voici l'implémentation de cet algorithme en langage machine :

In [ ]:
# À COMPLÉTER / REMPLACER PAR VOTRE DÉFI
notre_super_algo_creatif = [
    0b11_0010_00111_00000_0000000000000000, # R7 := i_GPIO
    0b01_1011_00111_00111_0000000000000111, # R7 := R7 × $7
    0b11_0011_00111_00000_0000000000000000, # o_GPIO := R7
    NOP,
    STOP
]

Cet algorithme réalise la fonction suivante : À COMPLÉTER, comme le prouve cette cellule de test :

In [ ]:
flash(notre_super_algo_creatif, ol, in_buf)
# À COMPLÉTER / REMPLACER
# Votre vecteur de test
nos_valeurs = [i for i in range(20)]

# Les résultats de votre programme
out = []
for t in nos_valeurs:
    out.append(test_PolyRISC(t, ol))
print(out)